# WAHIS Analytical Table Construction and Validation
## Machine Learning-Based Early Warning System for Poultry Disease Outbreaks in Nigeria

**Stage:** WAHIS data audit and analytical validation

This notebook documents the evidence-based validation performed before geographic and climate-data integration.


## Research decision rules

- WAHIS `-` values are treated as missing, not zero.
- Missing `New outbreaks` is not interpreted as disease absence.
- A raw WAHIS row is not automatically a final modelling observation.
- Multiple source records can occur for the same Administrative Division × Semester × Disease.
- Quantitative fields must be validated before aggregation.
- The final early-warning target is not fixed until WAHIS structure is fully validated.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'Quantitative data 2026-10-01 (1).csv'
df = pd.read_csv(RAW_PATH)
df['New outbreaks numeric'] = pd.to_numeric(df['New outbreaks'], errors='coerce')
print('Rows:', len(df))
print('Columns:', len(df.columns))


## 1. Basic WAHIS structure


In [ ]:
print('Shape:', df.shape)
print('Years:', sorted(df['Year'].dropna().unique().tolist()))
print('Diseases:', df['Disease'].nunique())
print('Administrative divisions:', df['Administrative Division'].nunique())
print('Exact duplicate rows:', df.duplicated().sum())


In [ ]:
candidate = df[df['New outbreaks numeric'].notna() & df['Administrative Division'].notna()].copy()
candidate['analytical_key'] = (candidate['Year'].astype(str)+'|'+candidate['Semester'].astype(str)+'|'+candidate['Administrative Division'].astype(str)+'|'+candidate['Disease'].astype(str))
print('Candidate rows:', len(candidate))
print('Unique candidate keys:', candidate['analytical_key'].nunique())
print('Duplicate candidate keys:', candidate['analytical_key'].duplicated().sum())


## 2. Coverage and candidate analytical unit

The export is semester-based (`Jan-Jun` and `Jul-Dec`). The candidate analytical dimension is Administrative Division × Semester × Disease. The 444-row candidate subset is exploratory only: multiple raw records can belong to the same disease/location/semester, and missing combinations cannot safely be filled with zero.


In [ ]:
display(pd.crosstab(candidate['Administrative Division'], candidate['Semester']))
display(pd.crosstab(candidate['Semester'], candidate['Disease']))


## 3. Missing `New outbreaks`

There are 735 records with `New outbreaks = '-'`. These records may still contain cases, deaths, or other disease-activity measures. Therefore `-` is missingness, not zero.


In [ ]:
missing_outbreak_records = df[df['New outbreaks numeric'].isna()].copy()
print('Records without numeric outbreak count:', len(missing_outbreak_records))
display(missing_outbreak_records[['Year','Semester','Administrative Division','Disease','Animal Category','Species','New outbreaks']].head(30))


## 4. Raw record structure

Inspection of Borno (Jan-Jun 2020) and Rivers (Jan-Jun 2022) showed multiple source records for the same disease/location/semester. One record can contain `New outbreaks`, while another contains cases/deaths and other quantitative measures. These records must not be blindly summed as independent outbreaks.


In [ ]:
cols=['Year','Semester','Administrative Division','Disease','Serotype/Subtype/Genotype','Animal Category','Species','Event_id','Outbreak_id','New outbreaks','Cases','Deaths','Killed and disposed of','Slaughtered','Vaccinated']
def inspect_group(year, semester, division):
    m=(df['Year']==year)&(df['Semester']==semester)&(df['Administrative Division']==division)
    return df.loc[m, cols]
display(inspect_group(2020,'Jan-Jun 2020','Borno'))
display(inspect_group(2022,'Jan-Jun 2022','Rivers'))


## 5. Quantitative-field validation

Original missing markers were checked for `New outbreaks`, `Cases`, `Deaths`, `Killed and disposed of`, `Slaughtered`, and `Vaccinated`. Numeric conversion with `errors='coerce'` creates missing values for `-` and produced no negative converted values in the validation run.


In [ ]:
numeric_columns=['New outbreaks','Cases','Deaths','Killed and disposed of','Slaughtered','Vaccinated']
for col in numeric_columns:
    print(f'===== {col} =====')
    print(df[col].astype(str).value_counts().loc[lambda s: s.index == '-'])
    converted=pd.to_numeric(df[col], errors='coerce')
    print('negative values after conversion:', int((converted < 0).sum()))


## 6. Disease activity versus outbreak-count availability

Examples show that a disease can have cases/deaths while `New outbreaks` is missing. Therefore `New outbreaks` alone should not be used as the sole disease-occurrence indicator. An operational target may use validated disease-activity evidence, but that definition is deliberately deferred until the final WAHIS panel is designed.


In [ ]:
disease_structure=(df[df['Year']>=2020]
 .groupby(['Year','Semester','Administrative Division','Disease'])
 .agg(source_records=('Disease','size'), outbreak_values=('New outbreaks numeric','count'), outbreak_total=('New outbreaks numeric','sum'), case_values=('Cases',lambda s: pd.to_numeric(s,errors='coerce').notna().sum()), death_values=('Deaths',lambda s: pd.to_numeric(s,errors='coerce').notna().sum()))
 .reset_index())
display(disease_structure.head(30))


## 7. Current methodological conclusion

1. Numeric `New outbreaks` observations in this export begin in 2020.
2. The export is semester-based.
3. Four diseases have numeric outbreak observations in the candidate period.
4. `-` is a missing-value marker, not zero.
5. Disease activity can be recorded in cases/deaths when outbreak count is missing.
6. Multiple source records can occur within the same disease/location/semester.
7. The 444-row outbreak-count subset is not yet the final modelling panel.
8. Missing geography–time–disease combinations must not automatically be labelled zero.


## 8. Next stage

**Stage 8 — Administrative Boundaries** is the next acquisition stage. Administrative boundaries will standardise the geographic reference layer before climate and environmental datasets are joined. No climate integration should occur until geographic names and spatial units are validated.
